# Fatal Force: deaths involving police in the United States

Day 100 assignment of the Udemy course *100 Days of Code: The Complete Python Pro Bootcamp*. The Washington Post has recorded every fatal shooting by a police officer in the line of duty since 1 January 2015. This notebook combines that database with US census data.

The topic is sensitive. The charts show counts and rates; they do not show causes. Read each chart with its caption.

The logic lives in the package `fatal_force` (`src/fatal_force/`); this notebook calls it and shows the result. Each step saves a screenshot in `docs/screenshots/`.

In [1]:
import polars as pl
from IPython.display import display

from fatal_force.infrastructure.cleaning import clean_all
from fatal_force.infrastructure.csv_reader import load_datasets
from fatal_force.infrastructure.exploration import summarise_frame
from fatal_force.presentation.screenshots import save_figure, summary_table_figure

pl.Config.set_tbl_rows(8)

polars.config.Config

## Load the data

The five course files are read from the git-ignored `data/` folder (see the README for how to obtain them).

In [2]:
raw = load_datasets()
raw_frames = {
    "Median household income": raw.hh_income,
    "Poverty rate": raw.pct_poverty,
    "High school completion": raw.pct_completed_hs,
    "Share of race by city": raw.share_race_city,
    "Deaths by police": raw.fatalities,
}

## Step 1: Preliminary data exploration

What is the shape of each frame, what are its columns, and are there missing values or duplicates?

In [3]:
raw_summaries = [summarise_frame(name, frame) for name, frame in raw_frames.items()]
for summary in raw_summaries:
    print(
        f"{summary.name}: {summary.rows} rows x {summary.columns} columns, "
        f"{summary.null_total} missing, {summary.duplicate_rows} duplicate rows"
    )
    print("   columns:", ", ".join(summary.column_names))
display(raw.fatalities.head())

Median household income: 29322 rows x 3 columns, 51 missing, 0 duplicate rows
   columns: Geographic Area, City, Median Income
Poverty rate: 29329 rows x 3 columns, 0 missing, 0 duplicate rows
   columns: Geographic Area, City, poverty_rate
High school completion: 29329 rows x 3 columns, 0 missing, 0 duplicate rows
   columns: Geographic Area, City, percent_completed_hs
Share of race by city: 29268 rows x 7 columns, 0 missing, 0 duplicate rows
   columns: Geographic area, City, share_white, share_black, share_native_american, share_asian, share_hispanic
Deaths by police: 2535 rows x 14 columns, 346 missing, 0 duplicate rows
   columns: id, name, date, manner_of_death, armed, age, gender, race, city, state, signs_of_mental_illness, threat_level, flee, body_camera


id,name,date,manner_of_death,armed,age,gender,race,city,state,signs_of_mental_illness,threat_level,flee,body_camera
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""3""","""Tim Elliot""","""02/01/15""","""shot""","""gun""","""53""","""M""","""A""","""Shelton""","""WA""","""TRUE""","""attack""","""Not fleeing""","""FALSE"""
"""4""","""Lewis Lee Lembke""","""02/01/15""","""shot""","""gun""","""47""","""M""","""W""","""Aloha""","""OR""","""FALSE""","""attack""","""Not fleeing""","""FALSE"""
"""5""","""John Paul Quintero""","""03/01/15""","""shot and Tasered""","""unarmed""","""23""","""M""","""H""","""Wichita""","""KS""","""FALSE""","""other""","""Not fleeing""","""FALSE"""
"""8""","""Matthew Hoffman""","""04/01/15""","""shot""","""toy weapon""","""32""","""M""","""W""","""San Francisco""","""CA""","""TRUE""","""attack""","""Not fleeing""","""FALSE"""
"""9""","""Michael Rodriguez""","""04/01/15""","""shot""","""nail gun""","""39""","""M""","""H""","""Evans""","""CO""","""FALSE""","""attack""","""Not fleeing""","""FALSE"""


In [4]:
figure = summary_table_figure("Step 1: raw data exploration", raw_summaries)
save_figure(figure, 1, "data-exploration")
figure

<Figure size 800x400 with 1 Axes>

## Step 2: Data cleaning

Census placeholders such as `(X)` and `-` become missing values, thousands separators and bounds such as `250,000+` are removed so the columns are numbers, the date column is parsed, and missing labels in the deaths data become `unknown`. Missing census values are kept as missing, not turned into zero, so they do not pull averages down.

In [5]:
clean = clean_all(raw)
clean_frames = {
    "Median household income": clean.hh_income,
    "Poverty rate": clean.pct_poverty,
    "High school completion": clean.pct_completed_hs,
    "Share of race by city": clean.share_race_city,
    "Deaths by police": clean.fatalities,
}
clean_summaries = [summarise_frame(name, frame) for name, frame in clean_frames.items()]
for before, after in zip(raw_summaries, clean_summaries, strict=True):
    print(f"{before.name}: missing {before.null_total} -> {after.null_total}")
display(clean.fatalities.head())

Median household income: missing 51 -> 1904
Poverty rate: missing 0 -> 201
High school completion: missing 0 -> 197
Share of race by city: missing 0 -> 100
Deaths by police: missing 346 -> 77


id,name,date,manner_of_death,armed,age,gender,race,city,state,signs_of_mental_illness,threat_level,flee,body_camera
i64,str,date,str,str,i64,str,str,str,str,bool,str,str,bool
3,"""Tim Elliot""",2015-01-02,"""shot""","""gun""",53,"""M""","""A""","""Shelton""","""WA""",true,"""attack""","""Not fleeing""",false
4,"""Lewis Lee Lembke""",2015-01-02,"""shot""","""gun""",47,"""M""","""W""","""Aloha""","""OR""",false,"""attack""","""Not fleeing""",false
5,"""John Paul Quintero""",2015-01-03,"""shot and Tasered""","""unarmed""",23,"""M""","""H""","""Wichita""","""KS""",false,"""other""","""Not fleeing""",false
8,"""Matthew Hoffman""",2015-01-04,"""shot""","""toy weapon""",32,"""M""","""W""","""San Francisco""","""CA""",true,"""attack""","""Not fleeing""",false
9,"""Michael Rodriguez""",2015-01-04,"""shot""","""nail gun""",39,"""M""","""H""","""Evans""","""CO""",false,"""attack""","""Not fleeing""",false


In [6]:
figure = summary_table_figure("Step 2: data after cleaning", clean_summaries)
save_figure(figure, 2, "data-cleaning")
figure

<Figure size 800x400 with 1 Axes>